# New hydropower workflows

RESKit's hydropower API separates discharge extraction from electricity generation. This notebook demonstrates the three new workflows:

1. `extract_discharge`: extract and resample discharge only.
2. `run_of_river_hydropower`: extract discharge and calculate run-of-river generation.
3. `release_generation`: calculate generation from a user-supplied turbine-release series.

All workflows use a `pandas.DatetimeIndex` to specify both the requested time range and time interval. Internally, discharge is represented as a mean flow rate in m³/s, and simulation arrays use the `(time, location)` orientation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from reskit.hydro.workflows import (
    extract_discharge,
    release_generation,
    run_of_river_hydropower,
)

## Placement and ParFlow configuration

Discharge extraction requires `lon` and `lat`. Hydropower calculation additionally requires `head` in metres and `capacity` in kW. The example finds the repository root so it works when started either from the repository root or from this notebook's directory.

ParFlow retrieval accesses a remote THREDDS service and can therefore take some time. Two products are registered: `parflow-1day` and `parflow-3hour`. The short name `parflow` remains an alias for `parflow-1day`.

In [2]:
placements = pd.DataFrame(
    {
        "hydro_plant_id": ["GHR03171", "GHR03272"],
        "lon": [11.273237, 11.601190],
        "lat": [48.750807, 48.778103],
        "head": [6.00, 6.07],  # m
        "capacity": [23_700.0, 23_300.0],  # kW
    }
)

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

parflow_root = repo_root / "reskit" / "hydro" / "_external_module" / "parflow_600m_runs"
parflow_options = {
    "root_dir": str(parflow_root),
    "alluvium_mask_file": str(parflow_root / "alluvium_mask.nc"),
    "indicator_file": str(
        parflow_root / "DE-0055_INDICATOR_regridded_rescaled_SoilGrids250-v2017_BGRvector_newAllv.nc"
    ),
    "fallback_mode": "max_annual",
}

placements

,hydro_plant_id,lon,lat,head,capacity
0,GHR03171,11.273237,48.750807,6.00,23700.0
1,GHR03272,11.601190,48.778103,6.07,23300.0


## 1. Extract discharge

The registered ParFlow products provide discharge volumes per native timestep: daily for `parflow-1day` and three-hourly for `parflow-3hour`. RESKit converts both to m³/s. When the requested interval differs from the selected product's native interval, the workflow emits a warning and resamples the discharge rate. Finer intervals are linearly interpolated; coarser intervals are time-averaged.

The following request uses the native daily interval, so no temporal resampling is necessary. The optional CSV records which alluvium-aware ParFlow grid cell was selected for each plant.

In [5]:
daily_time_index = pd.date_range("2020-01-01", "2020-12-31", freq="1D")

discharge = extract_discharge(
    placements=placements[["hydro_plant_id", "lon", "lat"]],
    product="parflow-1day",
    time_index=daily_time_index,
    product_options=parflow_options,
    output_selected_alluvium_candidate_path=(
        repo_root / "examples" / "7_hydro" / "outputs" / "selected_daily_parflow_candidates.csv"
    ),
)

discharge[["discharge_m3s"]]

/fast/home/s-chen/local_bin/python_envs_managed_by_mamba/reskit_hydropower/reskit/reskit/hydro/_external_module/parflow_600m_runs/parflow_data_extraction.py:137: RuntimeWarning: invalid value encountered in arccos
  spherical_distance = Rearth * np.arccos(


----------------------------------------
JSON record: 0
location ID, lon, lat, depth, sim-data: location_0, 11.274261474609375, 48.75282287597656, 0.2, https://service.tereno.net/thredds/dodsC/forecastnrw/products/ParFlow-DE06-HC_v03/sfd_DE05_ECMWF-HRES_hindcast_r1i1p2_FZJ-IBG3-ParFlowCLM380_hgfadapter-h00-v03bJuwelsGpuProdClimatologyTl_1day_20200101-20201231.nc
POI: index x-y and lon-lat on model grid: 1163, 653, 11.274261474609375, 48.75282287597656
ATTENTION: your chosen POI (point of interest) is located directly on a lake, river, or ocean grid element, it is recommended to specify an alternative, nearby longitude-latitude coordinate pair
indices of alternative close-by grid elements to POI:  [(653, 1164), (653, 1162), (654, 1163), (652, 1163), (652, 1164), (654, 1162), (652, 1162), (654, 1164), (653, 1161)]
alternative, recommended coordinate pairs close-by, not on river, lake, or ocean, Lon-Lat [dec deg], 5 digits, edit your JASON file accordingly:  11.28257 48.75332
alternative,

<xarray.Dataset> Size: 9kB
Dimensions:        (time: 366, location: 2)
Coordinates:
  * time           (time) datetime64[ns] 3kB 2020-01-01 ... 2020-12-31
  * location       (location) int64 16B 0 1
Data variables:
    discharge_m3s  (time, location) float64 6kB 420.7 463.1 ... 382.0 427.4
Attributes:
    discharge_product:            parflow-1day
    native_time_interval:         1 days 00:00:00
    requested_time_interval:      1 days 00:00:00
    temporal_resampling_applied:  False
    temporal_resampling_method:   none

Use `parflow-3hour` to obtain native three-hour discharge without interpolation. Alternatively, use `parflow-1day` with a three-hour `DatetimeIndex` to obtain interpolated values; the workflow will warn that interpolation cannot add real sub-daily hydrological information.

In [ ]:
three_hour_time_index = pd.date_range("2020-01-01 00:00", "2020-12-31 21:00", freq="3h")
three_hour_discharge = extract_discharge(
    placements=placements[["hydro_plant_id", "lon", "lat"]],
    product="parflow-3hour",
    time_index=three_hour_time_index,
    product_options=parflow_options,
)

three_hour_discharge[["discharge_m3s"]]

## 2. Calculate run-of-river generation

This workflow performs the complete chain: spatially select and extract ParFlow discharge, align it to `time_index`, and calculate hydropower. Generation is capped by installed capacity by default.

In [ ]:
run_of_river = run_of_river_hydropower(
    placements=placements,
    product="parflow-1day",
    time_index=daily_time_index,
    product_options=parflow_options,
    efficiency=0.88,
    cap_production_by_capacity=True,
    output_selected_alluvium_candidate_path=(repo_root / "outputs" / "selected_run_of_river_candidates.csv"),
)

run_of_river[
    [
        "discharge_m3s",
        "potential_power_kw",
        "power_kw",
        "generation_kwh",
        "capacity_factor",
        "usable_discharge_m3s",
        "spilled_discharge_m3s",
    ]
]

The key outputs are:

- `potential_power_kw`: hydraulic potential before the capacity limit.
- `power_kw`: actual power after applying the optional capacity limit.
- `generation_kwh`: power multiplied by the interval derived from `time_index`.
- `capacity_factor`: actual power divided by installed capacity.
- `usable_discharge_m3s`: flow required to produce the actual power.
- `spilled_discharge_m3s`: available flow that cannot be used because of the capacity limit.

In [ ]:
annual_generation_mwh = run_of_river["generation_kwh"].sum("time") / 1_000
annual_generation_mwh.rename("annual_generation_mwh")

## 3. Calculate generation from prescribed release

`release_generation` does not extract discharge and does not simulate reservoir storage or dispatch. Its input is interpreted as turbine release in m³/s. This makes it suitable when release has already been calculated by a reservoir-operation model or supplied as observations.

The release array must have shape `(time, location)`.

In [ ]:
release_time_index = pd.date_range("2020-01-01", periods=8, freq="3h")
release_m3s = np.array(
    [
        [120.0, 100.0],
        [130.0, 105.0],
        [140.0, 110.0],
        [150.0, 115.0],
        [145.0, 110.0],
        [135.0, 105.0],
        [125.0, 100.0],
        [115.0, 95.0],
    ]
)

release_generation_result = release_generation(
    placements=placements,
    discharge_m3s=release_m3s,
    time_index=release_time_index,
    efficiency=0.88,
    cap_production_by_capacity=True,
)

release_generation_result[["power_kw", "generation_kwh", "capacity_factor"]]

For a three-hour interval, `generation_kwh` equals `power_kw × 3`. Set `cap_production_by_capacity=False` to inspect unconstrained hydraulic potential; in that case, power and capacity factor can exceed the installed capacity.

In [ ]:
np.testing.assert_allclose(
    release_generation_result["generation_kwh"],
    release_generation_result["power_kw"] * 3.0,
)